In [0]:
from pyspark.sql.functions import *


orders = spark.read.table("project.silver.silver_orders")
customers = spark.read.table("project.silver.silver_customers")
order_items = spark.read.table("project.silver.silver_items")
payments = spark.read.table("project.silver.silver_payments")
reviews = spark.read.table("project.silver.silver_reviews")
products = spark.read.table("project.silver.silver_product")

In [0]:
TRAIN_CUTOFF = "2018-06-30"
PREDICTION_END = "2018-10-17"

### Map Orders to Customers

In [0]:
orders_customers = orders.join(
    customers.select("customer_id","customer_unique_id","customer_state"),
    on = "customer_id",
    how = "left"
)

In [0]:
payments_agg = payments.groupBy("order_id").agg(
    sum("payment_value").alias("payment_value"),
    avg("payment_installments").alias("payment_installments")
)

In [0]:
orders_full = orders_customers \
    .join(order_items, "order_id", "left") \
    .join(payments_agg, "order_id", "left") \
    .join(reviews, "order_id", "left") \
    .join(products.select("product_id","product_category_name_english"), "product_id", "left")

In [0]:
orders_train = orders_full.filter(
    col("order_purchase_timestamp") <= TRAIN_CUTOFF
)

In [0]:
customer_features = orders_train.groupBy("customer_unique_id").agg(

    countDistinct("order_id").alias("frequency"),

    sum("payment_value").alias("monetary"),

    countDistinct("product_id").alias("total_products_purchased"),

    countDistinct("product_category_name_english").alias("category_diversity"),

    avg("review_score").alias("avg_review_score"),

    min("order_purchase_timestamp").alias("first_order_date"),

    max("order_purchase_timestamp").alias("last_order_date"),

    avg("payment_value").alias("avg_payment_value"),

    avg("payment_installments").alias("avg_installments")
)

In [0]:
customer_features = customer_features.withColumns({

    "recency":
        datediff(lit(TRAIN_CUTOFF), col("last_order_date")),

    "avg_order_value":
        col("monetary") / col("frequency"),

    "days_as_customer":
        datediff(col("last_order_date"), col("first_order_date")),

    "feature_snapshot_date":
        lit(TRAIN_CUTOFF)
})

In [0]:
orders_future = orders_customers.filter(
    (col("order_purchase_timestamp") > TRAIN_CUTOFF) &
    (col("order_purchase_timestamp") <= PREDICTION_END)
)

active_customers = orders_future.select("customer_unique_id").distinct()

In [0]:
all_customers = orders_customers.select("customer_unique_id").distinct()

In [0]:
churn_customers = all_customers.join(
    active_customers,
    "customer_unique_id",
    "left_anti"
).withColumn("churn_label", lit(1))

In [0]:
active_label = active_customers.withColumn("churn_label", lit(0))

In [0]:
churn_labels = churn_customers.union(active_label)

In [0]:
orders_clv = orders_future.join(
    payments_agg,
    "order_id",
    "left"
)

In [0]:
clv_labels = orders_clv.groupBy("customer_unique_id").agg(
    sum("payment_value").alias("clv_label")
)

In [0]:
clv_labels = all_customers.join(
    clv_labels,
    "customer_unique_id",
    "left"
).fillna({"clv_label":0})

In [0]:
ml_dataset = customer_features \
    .join(churn_labels,"customer_unique_id","inner") \
    .join(clv_labels,"customer_unique_id","left")

In [0]:
ml_dataset = ml_dataset.fillna({
    "avg_review_score":0,
    "avg_payment_value":0,
    "avg_installments":0
})

In [0]:
display(ml_dataset.head(10))

customer_unique_id,frequency,monetary,total_products_purchased,category_diversity,avg_review_score,first_order_date,last_order_date,avg_payment_value,avg_installments,recency,avg_order_value,days_as_customer,feature_snapshot_date,churn_label,clv_label
809988e0e5fdc4bf04b87fbfdeb9d249,1,26.0,1,1,5.0,2017-07-31T15:00:07.000Z,2017-07-31T15:00:07.000Z,26.0,1.0,334,26.0,0,2018-06-30,1,0.0
ffebb6424578e7bb153322da9d65634f,1,665.7,1,1,1.0,2017-01-16T14:04:11.000Z,2017-01-16T14:04:11.000Z,665.7,8.0,530,665.7,0,2018-06-30,1,0.0
a859c8884a22c3c58cf229abd2a4d2f3,1,47.49,1,1,5.0,2017-09-14T16:29:22.000Z,2017-09-14T16:29:22.000Z,47.49,1.0,289,47.49,0,2018-06-30,1,0.0
fa08098a16131e9dd1dec2dd17fd4ee8,1,245.12,1,1,5.0,2017-05-02T14:07:29.000Z,2017-05-02T14:07:29.000Z,122.56,6.0,424,245.12,0,2018-06-30,1,0.0
bbf8e004f0bc05caf5d2214ef1245220,1,117.91,1,1,5.0,2017-12-04T18:10:46.000Z,2017-12-04T18:10:46.000Z,117.91,1.0,208,117.91,0,2018-06-30,1,0.0
5b7457a521492ba35573e2ea4c2e364a,1,217.67,1,1,4.0,2018-03-02T15:05:43.000Z,2018-03-02T15:05:43.000Z,217.67,2.0,120,217.67,0,2018-06-30,1,0.0
5c94dfc9ac107651749491c1d83a947a,1,71.56,1,1,4.0,2017-04-26T22:51:19.000Z,2017-04-26T22:51:19.000Z,71.56,1.0,430,71.56,0,2018-06-30,1,0.0
6262c41360266ae4843f72f8cb345640,1,43.09,1,1,4.0,2017-05-31T14:41:11.000Z,2017-05-31T14:41:11.000Z,43.09,1.0,395,43.09,0,2018-06-30,1,0.0
2526d3bb06624f7c5bedf2ef7a5bc4f0,1,476.38,1,1,5.0,2017-02-18T18:28:46.000Z,2017-02-18T18:28:46.000Z,476.38,3.0,497,476.38,0,2018-06-30,1,0.0
b986b28a12282612297a578e8d0536ef,1,35.38,1,1,2.0,2018-05-26T12:16:34.000Z,2018-05-26T12:16:34.000Z,35.38,1.0,35,35.38,0,2018-06-30,1,0.0


---

In [0]:
orders.select(
    min("order_purchase_timestamp"),
    max("order_purchase_timestamp")
).show()

+-----------------------------+-----------------------------+
|min(order_purchase_timestamp)|max(order_purchase_timestamp)|
+-----------------------------+-----------------------------+
|          2016-09-04 21:15:19|          2018-10-17 17:30:18|
+-----------------------------+-----------------------------+



In [0]:
display(orders.groupBy("customer_id") \
    .agg(count("*").alias("orders_per_customer")) \
    .selectExpr(
        "percentile_approx(orders_per_customer,0.25)",
        "percentile_approx(orders_per_customer,0.50)",
        "percentile_approx(orders_per_customer,0.75)",
        "max(orders_per_customer)"
    ))

"percentile_approx(orders_per_customer, 0.25, 10000)","percentile_approx(orders_per_customer, 0.50, 10000)","percentile_approx(orders_per_customer, 0.75, 10000)",max(orders_per_customer)
1,1,1,1


In [0]:
from pyspark.sql.functions import *

customer_last_order = orders.groupBy("customer_id").agg(
    max("order_purchase_timestamp").alias("last_order_date")
)

customer_last_order = customer_last_order.withColumn(
    "days_since_last_order",
    datediff(current_date(), col("last_order_date"))
)

display(customer_last_order.selectExpr(
    "percentile_approx(days_since_last_order,0.50)",
    "percentile_approx(days_since_last_order,0.75)",
    "percentile_approx(days_since_last_order,0.90)",
    "max(days_since_last_order)"
))

"percentile_approx(days_since_last_order, 0.50, 10000)","percentile_approx(days_since_last_order, 0.75, 10000)","percentile_approx(days_since_last_order, 0.90, 10000)",max(days_since_last_order)
2973,3101,3219,3474


In [0]:
display(payments.selectExpr(
    "percentile_approx(payment_value,0.25)",
    "percentile_approx(payment_value,0.50)",
    "percentile_approx(payment_value,0.75)",
    "max(payment_value)"
))

"percentile_approx(payment_value, 0.25, 10000)","percentile_approx(payment_value, 0.50, 10000)","percentile_approx(payment_value, 0.75, 10000)",max(payment_value)
56.84,100.0,171.81,13664.08
